In [3]:
import numpy as np
import pandas as pd
from graph_tool.all import *
from pyvis.network import Network

#############################
# Parameters
#############################

# Distance cutoff
distance_cutoff = 130  # Only include edges below this distance

#############################
# Load Data
#############################

# Load the distance matrix
distance_matrix_file = "combined_distance_matrix.txt"
distance_matrix = np.loadtxt(distance_matrix_file)

# Load segment order mapping
segment_mapping_file = "segment_order_mapping.txt"
segment_data = pd.read_csv(segment_mapping_file)

# Extract segment IDs and types
segment_ids = segment_data["SegmentID"].values
component_types = segment_data["ComponentType"].values

# Create a lookup dictionary for component types
segment_dict = dict(zip(segment_ids, component_types))

#############################
# Create the Graph
#############################

# Initialize the graph
g = Graph(directed=False)

# Add vertices for each segment
vprop_type = g.new_vertex_property("string")  # Protein type (spike, mucin, albumin)
vertices = {}

for segment_id in segment_ids:
    v = g.add_vertex()
    vertices[segment_id] = v
    vprop_type[v] = segment_dict[segment_id]

# Add edges based on the distance matrix
edge_data = []  # Store edge information for pyvis

for i in range(len(segment_ids)):
    for j in range(i + 1, len(segment_ids)):
        distance = distance_matrix[i, j]
        if 0 < distance < distance_cutoff:
            g.add_edge(vertices[segment_ids[i]], vertices[segment_ids[j]])
            edge_data.append((segment_ids[i], segment_ids[j], distance))

#############################
# Generate Node Positions
#############################

# Generate circular positions for nodes
num_vertices = g.num_vertices()
theta = np.linspace(0, 2 * np.pi, num_vertices, endpoint=False)
positions = {}

for i, v in enumerate(g.vertices()):
    positions[v] = [np.cos(theta[i]), np.sin(theta[i])]

#############################
# Convert to pyvis
#############################

# Initialize pyvis network
net = Network(notebook=True, height="750px", width="100%", bgcolor="#222222", font_color="white")

# Add nodes to pyvis
for segment_id, v in vertices.items():
    node_type = vprop_type[v]
    color = {
        "spike": "SlateBlue",
        "mucin": "Crimson",
        "albumin": "MediumSeaGreen"
    }[node_type]
    net.add_node(
        segment_id,
        label=segment_id,
        title=f"SegmentID: {segment_id}<br>Type: {node_type}",
        x=positions[v][0] * 300,  # Scale for pyvis
        y=positions[v][1] * 300,
        color=color,
    )

# Add edges to pyvis
for segment_id1, segment_id2, distance in edge_data:
    net.add_edge(
        segment_id1,
        segment_id2,
        title=f"Connection: {segment_id1} ↔ {segment_id2}<br>Distance: {distance:.2f}",
    )

#############################
# Render Interactive Graph
#############################

net.show("interactive_graph_with_distances.html")


interactive_graph_with_distances.html
